# 04f · Feature representation guide (Section 9)

**Purpose.** Fill the decision table (situation → recommended representation → evidence) from the results, and write the data setup checklist: baseline correction, split by session, train only fitting, baseline only sanity check.

**Inputs**
- `results/*.csv` and `report/tables/` from 04a to 04e

**Outputs**
- `report/tables/` guide table

In [ ]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from sklearn.ensemble import RandomForestClassifier
from src.cv import load_baseline_level, load_processed, make_folds
from src.features.handcrafted import FEATURE_NAMES, handcrafted_features

TABLES = ROOT / "report/tables"
FIG = ROOT / "report/figures"
RESULTS = ROOT / "results"
MAIN = ["01a_gwt_raw", "01b_es_raw", "02b_gwt_handcrafted", "02c_es_handcrafted",
        "03a_gwt_encoder", "03b_es_encoder", "03c_end_to_end_reference"]
missing = [n for n in MAIN if not (RESULTS / f"{n}.csv").exists()]
assert not missing, f"run these notebooks first: {missing}"
scores = pd.concat([pd.read_csv(RESULTS / f"{n}.csv") for n in MAIN], ignore_index=True)
REPS = ["R1_raw", "R2_handcrafted", "R3_encoder"]
HEADS = ["logreg", "svm", "rf", "mlp"]
cost = pd.read_csv(TABLES / "C4_cost.csv")
sanity = pd.read_csv(RESULTS / "04d_baseline_sanity_check.csv")
eff = pd.read_csv(RESULTS / "04b_label_efficiency.csv")

## Best representation per dataset and task
Score = best head (highest mean macro F1 over 3 seeds × 5 folds).

In [ ]:
mean_f1 = scores[scores["rep"].isin(REPS)].groupby(["dataset", "task", "rep", "head"])["macro_f1"].mean()
best_rep = mean_f1.groupby(["dataset", "task", "rep"]).max().unstack("rep")[REPS]
best_rep["winner"] = best_rep.idxmax(axis=1)
best_rep.round(3)

## Interpretability: random forest importance of the R2 features
Fitted on the **training part** of fold 0, seed 0 only (no test data).

In [ ]:
importance = {}
for dataset in ["gwt", "es"]:
    X, meta, info = load_processed(dataset)
    F = handcrafted_features(X, load_baseline_level(dataset), info["n_base"], info["exposure_end"])  # (N, C * 12)
    tr, _ = make_folds(dataset, meta, seed=0)[0]
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0).fit(F[tr], meta["y_multi"].iloc[tr])
    names = [f"{info.get('channels', [f's{c}' for c in range(X.shape[1])])[c]}:{f}" for c in range(X.shape[1]) for f in FEATURE_NAMES]
    imp = pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False)
    importance[dataset] = imp
    by_feature = imp.groupby(lambda n: n.split(":")[1]).sum().sort_values(ascending=False)
    print(dataset, "- top 10 sensor:feature"); display(imp.head(10).round(4))
    print(dataset, "- importance summed per feature type"); display(by_feature.round(3))

## The guide (Section 9), filled from the results

In [ ]:
def winner(dataset, task):
    return best_rep.loc[(dataset, task), "winner"]


def f1(dataset, task, rep):
    return best_rep.loc[(dataset, task), rep]


# cheapest representation within 0.02 macro F1 of the best, per dataset (multiclass)
cheap = {}
for dataset in ["gwt", "es"]:
    ok = [r for r in REPS if f1(dataset, "multi", r) >= f1(dataset, "multi", winner(dataset, "multi")) - 0.02]
    c = cost[(cost["dataset"] == dataset) & cost["rep"].isin(ok)]
    cheap[dataset] = c.assign(total=c["build_features_s"] + c["train_best_head_s"]).sort_values("total")["rep"].iloc[0]

low = eff[eff["train_fraction"] == eff["train_fraction"].min()]
low_winner = low.groupby(["rep", "head"])["macro_f1"].mean().groupby("rep").max().idxmax()

guide = pd.DataFrame([
    ("Lab data, many labeled recordings", winner("gwt", "multi"), "C1 GasWindTunnel, C3",
     f"GWT multiclass best: {f1('gwt', 'multi', winner('gwt', 'multi')):.3f}; with 10% labels the best is {low_winner}"),
    ("Real world scents, few samples", winner("es", "multi"), "C1 EverydayScents, C2",
     f"ES multiclass best: {f1('es', 'multi', winner('es', 'multi')):.3f}"),
    ("Need interpretable features", "R2_handcrafted", "C1, random forest importance",
     f"R2 vs best: GWT {f1('gwt', 'multi', 'R2_handcrafted'):.3f} vs {f1('gwt', 'multi', winner('gwt', 'multi')):.3f}, "
     f"ES {f1('es', 'multi', 'R2_handcrafted'):.3f} vs {f1('es', 'multi', winner('es', 'multi')):.3f}"),
    ("No GPU, limited time", f"GWT: {cheap['gwt']}, ES: {cheap['es']}", "C4",
     "cheapest representation within 0.02 macro F1 of the best"),
    ("Detecting one target (binary)", f"GWT: {winner('gwt', 'binary')}, ES: {winner('es', 'binary')}", "C1 binary tables", ""),
    ("Identifying many classes", winner("gwt", "multi"), "C1 multiclass tables", "10-class GasWindTunnel task"),
], columns=["Your situation", "Recommended representation", "Evidence", "Numbers"])
guide.to_csv(TABLES / "guide_table.csv", index=False)
guide

## Data setup checklist

In [ ]:
chance = {("gwt", "multi"): 0.10, ("es", "multi"): 0.25, ("gwt", "binary"): 0.5, ("es", "binary"): 0.5}
leak = sanity.groupby(["dataset", "task", "rep", "head"])["balanced_acc"].mean().groupby(["dataset", "task"]).max()
checklist = [
    "Baseline correction: subtract the per-sensor resting level before building features.",
    "Split by session (grouped folds) whenever several samples come from one session.",
    "Fit every scaler, encoder and head on the training fold only; tune on an inner split.",
    "Run the baseline only sanity check: " + "; ".join(
        f"{d} {t} {v:.2f} (chance {chance[(d, t)]:.2f})" for (d, t), v in leak.items()),
]
for item in checklist:
    print("-", item)